In [66]:
import time

class BranchAndBound(object):
    def __init__(self, verbose=False, increase_rate=None):
        self.bound = float('-inf')
        self.best_score = None
        self.best_config = None
        self.nb_cut = 0
        self.nb_full = 0
        self.avg_stack = 0
        self.last_nb_cut = 0
        self.sum_stack = 0
        self.verbose = verbose or bool(decrease_rate)
        self.increase_rate = increase_rate
        self.last_update = time.time()
        
    @staticmethod
    def get_children(config):
        pass

    @staticmethod
    def get_score(config):
        pass

    def run(self, config, stack=0):
        score = self.get_score(config)
        if score <= self.bound:
            if self.verbose:
                self.nb_cut += 1
                self.sum_stack += stack
                self.periodic_exec()
            return(None)
        has_children = False
        for child in self.get_children(config):
            has_children = True
            self.run(child, stack=stack+1)
        if not has_children:
            if self.verbose:
                self.nb_full += 1
                self.periodic_exec()
            if score > self.bound:
                self.best_score = score
                self.best_config = config
                self.bound = score

    def print_stats(self):
        print(f'#full={self.nb_full}, #cut={self.nb_cut}, avg_stack={self.avg_stack}, best={self.best_score}, bound={self.bound}')

    def update_stats(self):
        nb_cut_interval = self.nb_cut - self.last_nb_cut
        self.avg_stack = self.sum_stack/nb_cut_interval if nb_cut_interval > 0 else float('nan')
        self.sum_stack = 0
        self.last_nb_cut = self.nb_cut
        if self.increase_rate:
            self.bound *= self.increase_rate
        
    def periodic_exec(self, force_refresh=False):
        current_time = time.time()
        if not force_refresh and current_time < self.last_update + 1:
            return
        self.update_stats()
        self.print_stats()
        self.last_update = current_time

# Usage example with min-max cost assignation 

In [72]:
import numpy as np
nb_jobs = 25
nb_workers = 25
cost = np.random.rand(nb_jobs, nb_workers)

def get_children(config):
    nb_assign, jobs, workers, unassigned_workers = config
    if not unassigned_workers:
        return([])
    jobs = jobs + [nb_assign]
    nb_assign = nb_assign + 1
    for w in unassigned_workers:
        yield((nb_assign, jobs, workers+[w], unassigned_workers.difference([w])))

def get_score(config):
    _, jobs, workers, _ = config
    if len(jobs) == 0:
        return(0)
    score = cost[jobs,workers].max()
    return(-score)

# Quick exploration (retrieval of bound guess)
bnb = BranchAndBound(verbose=True, increase_rate=0.99)
bnb.get_score = get_score
bnb.get_children = get_children
bnb.bound = -0.5 # Initial guess
bnb.run((0,[],[],set(range(nb_workers))))
print(bnb.best_score)

# Exhaustive search using best previous score as guess
bnb.bound = bnb.best_score
bnb.increase_rate = None
bnb.run((0,[],[],set(range(nb_workers))))
print(bnb.best_score)

#full=9, #cut=115619, avg_stack=21.15695517172783, best=-0.3207256530369793, bound=-0.3175183965066095
#full=9, #cut=235192, avg_stack=21.040326829635454, best=-0.3207256530369793, bound=-0.3143432125415434
#full=9, #cut=356576, avg_stack=21.029715613260397, best=-0.3207256530369793, bound=-0.31119978041612795
#full=9, #cut=477297, avg_stack=21.043115945030276, best=-0.3207256530369793, bound=-0.3080877826119667
#full=9, #cut=597729, avg_stack=21.000307227314998, best=-0.3207256530369793, bound=-0.305006904785847
#full=9, #cut=721924, avg_stack=20.593824228028502, best=-0.3207256530369793, bound=-0.30195683573798854
#full=9, #cut=837666, avg_stack=20.58932798811149, best=-0.3207256530369793, bound=-0.29893726738060866
#full=9, #cut=958788, avg_stack=20.940192533148394, best=-0.3207256530369793, bound=-0.29594789470680255
#full=9, #cut=1079524, avg_stack=20.97544228730453, best=-0.3207256530369793, bound=-0.2929884157597345
#full=9, #cut=1198851, avg_stack=20.94178182640978, best=-0.320